# Chapter 11: A Pythonic Object
**Fluent Python, 2nd Edition** -- Luciano Ramalho

> "To build Pythonic objects, observe how real Python objects behave."

## TL;DR
Python's data model lets your classes behave like built-in types by implementing special ('dunder') methods. This chapter builds a **Vector2d** class step-by-step, demonstrating:
- **Object representations**: `__repr__`, `__str__`, `__format__`, `__bytes__`
- **Alternative constructors** via `@classmethod`
- **Hashability** through `__hash__` + `__eq__` + immutability
- **Read-only properties** and private attributes (name mangling)
- **Memory optimization** with `__slots__`
- **Overriding class attributes** idiomatically

**Related wiki articles:** [[object-representations]], [[classmethod-vs-staticmethod]], [[hashable-objects]], [[private-and-protected-attributes]], [[slots]], [[overriding-class-attributes]], [[read-only-properties]]

---
## 1. Object Representations: `__repr__`, `__str__`, `__format__`, `__bytes__`

Python has **multiple string protocols** for objects:

| Method | Called by | Audience |
|---|---|---|
| `__repr__` | `repr()`, debugger, console | Developer |
| `__str__` | `str()`, `print()` | End user |
| `__format__` | `format()`, f-strings | Configurable display |
| `__bytes__` | `bytes()` | Binary serialization |

> **Rule of thumb:** If you implement only one, choose `__repr__`, because Python falls back to it when `__str__` is missing.

See also: [[object-representations]]

In [ ]:
from array import array
import math

class Vector2d:
    """A two-dimensional Euclidean vector."""
    typecode = 'd'

    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def __iter__(self):
        return (i for i in (self.x, self.y))

    def __repr__(self):
        class_name = type(self).__name__
        return '{}({!r}, {!r})'.format(class_name, *self)

    def __str__(self):
        return str(tuple(self))

    def __bytes__(self):
        return (bytes([ord(self.typecode)]) +
                bytes(array(self.typecode, self)))

    def __eq__(self, other):
        return tuple(self) == tuple(other)

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __bool__(self):
        return bool(abs(self))

# Demonstrate the different representations
v = Vector2d(3, 4)
print(f"repr : {repr(v)}")
print(f"str  : {str(v)}")
print(f"bytes: {bytes(v)}")
print(f"abs  : {abs(v)}")
print(f"bool : {bool(v)}, bool(Vector2d(0,0))={bool(Vector2d(0,0))}")

### Key insight: `__iter__` enables unpacking
Because `Vector2d` is iterable, you can unpack it into variables and use `*self` inside `__repr__`.

In [ ]:
v = Vector2d(3, 4)
x, y = v  # unpacking works because of __iter__
print(f"Unpacked: x={x}, y={y}")

# repr produces valid constructor call
v_clone = eval(repr(v))
print(f"Clone via eval(repr()): {v_clone}")
print(f"Equal? {v == v_clone}")

---
## 2. Custom Formatted Displays with `__format__`

The **Format Specification Mini-Language** is extensible. Each class can define its own format codes. Our Vector2d supports:
- Standard float codes (`.2f`, `.3e`) applied to each component
- A custom `'p'` suffix for **polar coordinates**: `<magnitude, angle>`

In [ ]:
class Vector2dFmt:
    typecode = 'd'

    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def __iter__(self):
        return (i for i in (self.x, self.y))

    def __repr__(self):
        class_name = type(self).__name__
        return '{}({!r}, {!r})'.format(class_name, *self)

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def angle(self):
        return math.atan2(self.y, self.x)

    def __format__(self, fmt_spec=''):
        if fmt_spec.endswith('p'):
            fmt_spec = fmt_spec[:-1]
            coords = (abs(self), self.angle())
            outer_fmt = '<{}, {}>'
        else:
            coords = self
            outer_fmt = '({}, {})'
        components = (format(c, fmt_spec) for c in coords)
        return outer_fmt.format(*components)

v = Vector2dFmt(1, 1)
print(f"Default:    {format(v)}")
print(f"Fixed 2:    {format(v, '.2f')}")
print(f"Sci 3:      {format(v, '.3e')}")
print(f"Polar:      {format(v, 'p')}")
print(f"Polar .3e:  {format(v, '.3ep')}")
print(f"Polar .5f:  {format(v, '0.5fp')}")
print(f"f-string:   {v:.2f}")

---
## 3. `@classmethod` vs `@staticmethod`

| Feature | `@classmethod` | `@staticmethod` |
|---|---|---|
| First argument | `cls` (the class) | None (plain function) |
| Use case | Alternative constructors | Rarely needed |
| Inheritance | Respects subclass (`cls` is the actual subclass) | No class awareness |

> **Ramalho's advice:** "Good use cases for `staticmethod` are very rare. A module-level function is simpler."

See also: [[classmethod-vs-staticmethod]]

In [ ]:
class Demo:
    @classmethod
    def klassmeth(*args):
        return args

    @staticmethod
    def statmeth(*args):
        return args

# classmethod always receives the class as first arg
print(f"classmethod(): {Demo.klassmeth()}")
print(f"classmethod('spam'): {Demo.klassmeth('spam')}")

# staticmethod is just a plain function
print(f"staticmethod(): {Demo.statmeth()}")
print(f"staticmethod('spam'): {Demo.statmeth('spam')}")

In [ ]:
# Practical example: alternative constructor via @classmethod
from array import array

class Vector2dAlt:
    typecode = 'd'

    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def __iter__(self):
        return (i for i in (self.x, self.y))

    def __repr__(self):
        return f'Vector2dAlt({self.x!r}, {self.y!r})'

    def __bytes__(self):
        return (bytes([ord(self.typecode)]) +
                bytes(array(self.typecode, self)))

    @classmethod
    def frombytes(cls, octets):
        """Alternative constructor: build from binary representation."""
        typecode = chr(octets[0])
        memv = memoryview(octets[1:]).cast(typecode)
        return cls(*memv)  # cls, not Vector2dAlt -- respects subclasses!

v1 = Vector2dAlt(3, 4)
raw = bytes(v1)
print(f"bytes: {raw}")

v2 = Vector2dAlt.frombytes(raw)
print(f"Restored: {v2}")

---
## 4. Making Objects Hashable: `__hash__` and `__eq__`

To use objects as **dict keys** or **set members**, they must be hashable. Requirements:
1. Implement `__hash__` returning an `int`
2. Implement `__eq__` (already have it)
3. Objects that compare equal **must** have the same hash
4. The hash value should **never change** -- so make the object immutable

See also: [[hashable-objects]], [[read-only-properties]]

In [ ]:
import math
from array import array

class Vector2dHash:
    typecode = 'd'
    __match_args__ = ('x', 'y')

    def __init__(self, x, y):
        self.__x = float(x)   # private attributes for immutability
        self.__y = float(y)

    @property
    def x(self):               # read-only property
        return self.__x

    @property
    def y(self):               # read-only property
        return self.__y

    def __iter__(self):
        return (i for i in (self.x, self.y))

    def __repr__(self):
        class_name = type(self).__name__
        return '{}({!r}, {!r})'.format(class_name, *self)

    def __str__(self):
        return str(tuple(self))

    def __eq__(self, other):
        return tuple(self) == tuple(other)

    def __hash__(self):
        return hash((self.x, self.y))

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __bool__(self):
        return bool(abs(self))

# Now it is hashable!
v1 = Vector2dHash(3, 4)
v2 = Vector2dHash(3.1, 4.2)
print(f"hash(v1)={hash(v1)}, hash(v2)={hash(v2)}")
print(f"Set: { {v1, v2} }")

# Immutability enforced
try:
    v1.x = 99
except AttributeError as e:
    print(f"Cannot set x: {e}")

---
## 5. Private and Protected Attributes (Name Mangling)

Python has **no true private** attributes. Instead:

| Convention | Syntax | Mechanism |
|---|---|---|
| **Private** | `__attr` (2 underscores) | Name mangling: stored as `_ClassName__attr` |
| **Protected** | `_attr` (1 underscore) | Convention only -- not enforced |
| **Public** | `attr` | No protection |

Name mangling prevents **accidental** override in subclasses, not malicious access.

See also: [[private-and-protected-attributes]]

In [ ]:
v = Vector2dHash(3, 4)

# Name mangling in action
print(f"Instance __dict__: {v.__dict__}")

# You CAN access mangled names (safety, not security)
print(f"Direct access: v._Vector2dHash__x = {v._Vector2dHash__x}")

# The property provides the clean public API
print(f"Property access: v.x = {v.x}")

In [ ]:
# Name mangling prevents subclass collisions
class Parent:
    def __init__(self):
        self.__secret = "parent's secret"

    def reveal(self):
        return self.__secret

class Child(Parent):
    def __init__(self):
        super().__init__()
        self.__secret = "child's secret"  # does NOT clobber parent's!

    def reveal_child(self):
        return self.__secret

c = Child()
print(f"Parent's method sees: {c.reveal()}")      # parent's secret
print(f"Child's method sees:  {c.reveal_child()}")  # child's secret
print(f"Internal storage:      {c.__dict__}")

---
## 6. Saving Memory with `__slots__`

By default, Python stores instance attributes in a per-instance `__dict__`. With `__slots__`:
- Attributes are stored in a **hidden array** (no `__dict__`)
- **Memory savings**: ~3x for millions of instances
- **Tradeoff**: Cannot add arbitrary attributes, must redeclare in subclasses

See also: [[slots]]

In [ ]:
import sys

class RegularPoint:
    def __init__(self, x, y):
        self.x = x
        self.y = y

class SlottedPoint:
    __slots__ = ('x', 'y')

    def __init__(self, x, y):
        self.x = x
        self.y = y

# Compare memory usage
regular = RegularPoint(1.0, 2.0)
slotted = SlottedPoint(1.0, 2.0)

print(f"Regular has __dict__: {hasattr(regular, '__dict__')}")
print(f"Slotted has __dict__: {hasattr(slotted, '__dict__')}")

# Cannot add arbitrary attributes to slotted
try:
    slotted.z = 3.0
except AttributeError as e:
    print(f"Cannot add 'z': {e}")

# Memory comparison (rough estimate via sys.getsizeof)
print(f"\nRegular instance size: {sys.getsizeof(regular)} bytes")
print(f"Regular __dict__ size: {sys.getsizeof(regular.__dict__)} bytes")
print(f"Slotted instance size: {sys.getsizeof(slotted)} bytes")

In [ ]:
# __slots__ inheritance gotcha
class Pixel:
    __slots__ = ('x', 'y')

class OpenPixel(Pixel):
    pass  # No __slots__ declared!

class ColorPixel(Pixel):
    __slots__ = ('color',)  # Adds to parent's slots

op = OpenPixel()
print(f"OpenPixel has __dict__: {hasattr(op, '__dict__')}")  # True! Surprise!
op.color = 'red'  # This works -- stored in __dict__
print(f"OpenPixel can add attrs: color={op.color}")

cp = ColorPixel()
print(f"\nColorPixel has __dict__: {hasattr(cp, '__dict__')}")  # False
cp.x = 1
cp.y = 2
cp.color = 'blue'
print(f"ColorPixel: ({cp.x}, {cp.y}, {cp.color})")
try:
    cp.flavor = 'banana'
except AttributeError as e:
    print(f"Cannot add 'flavor': {e}")

---
## 7. Overriding Class Attributes

Class attributes work as **defaults** for instance attribute lookups. When you write to `self.attr`, you create an **instance attribute** that **shadows** the class attribute.

**Idiomatic pattern:** Subclass to customize class-wide defaults rather than mutating the base class.

See also: [[overriding-class-attributes]]

In [ ]:
class Vector2dDemo:
    typecode = 'd'  # class attribute: 8-byte double

    def __init__(self, x, y):
        self.x = float(x)
        self.y = float(y)

    def export_info(self):
        byte_size = {'d': 8, 'f': 4}[self.typecode]
        return f'typecode={self.typecode!r}, bytes per component: {byte_size}'

# Default behavior
v1 = Vector2dDemo(1.1, 2.2)
print(f"Default: {v1.export_info()}")

# Override per-instance (creates instance attribute)
v1.typecode = 'f'
print(f"After instance override: {v1.export_info()}")
print(f"Class attribute unchanged: {Vector2dDemo.typecode!r}")

# Idiomatic: subclass to customize
class ShortVector2d(Vector2dDemo):
    typecode = 'f'  # 4-byte float

sv = ShortVector2d(1.1, 2.2)
print(f"\nSubclass: {sv.export_info()}")
print(f"Base class still: {Vector2dDemo.typecode!r}")

In [ ]:
# Why use type(self).__name__ in __repr__?
class Base:
    def __repr__(self):
        # GOOD: uses runtime class name
        return f'{type(self).__name__}(...)'

class Sub(Base):
    pass

print(repr(Base()))  # Base(...)
print(repr(Sub()))   # Sub(...) -- correct! No override needed

---
## Exercises

Test your understanding of Chapter 11 concepts.

### Exercise 1: Implement `__format__` for a Money class

Create a `Money` class that:
- Stores `amount` (float) and `currency` (str, e.g. "USD")
- `__repr__` returns `Money(10.5, 'USD')`
- `__format__` with no spec returns `'10.50 USD'`
- `__format__` with a float spec (e.g. '.3f') applies it to the amount

In [ ]:
# Exercise 1: Your solution here
class Money:
    def __init__(self, amount, currency):
        self.amount = float(amount)
        self.currency = currency

    def __repr__(self):
        return f'Money({self.amount!r}, {self.currency!r})'

    def __format__(self, fmt_spec=''):
        if not fmt_spec:
            fmt_spec = '.2f'
        return f'{format(self.amount, fmt_spec)} {self.currency}'

# Test it
m = Money(10.5, 'USD')
print(f"repr: {repr(m)}")
print(f"format default: {format(m)}")
print(f"format .3f: {format(m, '.3f')}")
print(f"f-string: {m:.1f}")

### Exercise 2: Make a hashable `Color` class

Create a `Color` with read-only `r`, `g`, `b` components (0-255) that can be used in a set.

In [ ]:
# Exercise 2: Your solution here
class Color:
    __slots__ = ('__r', '__g', '__b')

    def __init__(self, r, g, b):
        for val in (r, g, b):
            if not (0 <= val <= 255):
                raise ValueError(f'Component must be 0-255, got {val}')
        object.__setattr__(self, '_Color__r', int(r))
        object.__setattr__(self, '_Color__g', int(g))
        object.__setattr__(self, '_Color__b', int(b))

    @property
    def r(self):
        return self.__r

    @property
    def g(self):
        return self.__g

    @property
    def b(self):
        return self.__b

    def __repr__(self):
        return f'Color({self.r}, {self.g}, {self.b})'

    def __eq__(self, other):
        if isinstance(other, Color):
            return (self.r, self.g, self.b) == (other.r, other.g, other.b)
        return NotImplemented

    def __hash__(self):
        return hash((self.r, self.g, self.b))

    @classmethod
    def from_hex(cls, hex_str):
        """Alternative constructor: Color.from_hex('#FF8800')"""
        hex_str = hex_str.lstrip('#')
        return cls(int(hex_str[0:2], 16), int(hex_str[2:4], 16), int(hex_str[4:6], 16))

# Test
red = Color(255, 0, 0)
also_red = Color(255, 0, 0)
blue = Color(0, 0, 255)
orange = Color.from_hex('#FF8800')

print(f"{red} == {also_red}? {red == also_red}")
print(f"hash equal? {hash(red) == hash(also_red)}")
print(f"Set: { {red, also_red, blue, orange} }")
print(f"From hex: {orange}")

# Immutability
try:
    red.r = 128
except AttributeError as e:
    print(f"Immutable: {e}")

### Exercise 3: `__slots__` inheritance quiz

Predict the output before running the cell.

In [ ]:
class A:
    __slots__ = ('x',)

class B(A):
    __slots__ = ('y',)

class C(A):
    pass  # no __slots__

b = B()
b.x = 1
b.y = 2
print(f"B instance: x={b.x}, y={b.y}")
print(f"B has __dict__? {hasattr(b, '__dict__')}")

c = C()
c.x = 1
c.z = 99  # Will this work?
print(f"C instance: x={c.x}, z={c.z}")
print(f"C has __dict__? {hasattr(c, '__dict__')}")
print(f"C.__dict__ contents: {c.__dict__}")  # x is NOT here (in slot)

---
## Key Takeaways

1. **`__repr__` is for developers, `__str__` is for users.** When in doubt, implement `__repr__`; Python falls back to it.

2. **`@classmethod` shines for alternative constructors** (like `frombytes`). Use `cls` not the class name, so subclasses inherit correctly.

3. **Hashability requires consistency:** `__hash__` and `__eq__` must agree. Make hashable objects immutable with read-only properties.

4. **Name mangling (`__attr`) is safety, not security.** It prevents accidental clobbering in subclasses. For simple "internal use" markers, prefer `_attr`.

5. **`__slots__` trades flexibility for memory.** Only use it when you have millions of instances. Remember to redeclare in subclasses.

6. **Subclass to override class attributes** rather than mutating the base class. Use `type(self).__name__` in `__repr__` so subclasses get correct names for free.

7. **"To build Pythonic objects, observe how real Python objects behave."** Implement only the special methods your use case demands.

**Related chapters:** [[01-python-data-model]], [[12-special-methods-for-sequences]], [[16-operator-overloading]]